# Same Anwsers on Test Anomaly

In [16]:
import pandas as pd
import pyodbc  
import matplotlib.pyplot as plt
import numpy as np

## Import Data from csv

In [17]:
fca_test = pd.read_csv("../../../decoded_data/SJT/FactTest.csv")
fca_question = pd.read_csv("../../../decoded_data/SJT/FactQuestionSJT.csv")

## Data Inspection

In [18]:
fca_question.head()

,QuestionKey,InstanceID,ItemID,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpent,Test,TestKey
0,1,1,1,5,4,2,181,SJT,160065
1,2,1,2,4,3,2,172,SJT,160065
2,3,1,3,4,2,3,136,SJT,160065
3,4,1,4,3,2,4,266,SJT,160065
4,5,1,5,2,5,1,198,SJT,160065


In [19]:
fca_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89622 entries, 0 to 89621
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   QuestionKey      89622 non-null  int64 
 1   InstanceID       89622 non-null  int64 
 2   ItemID           89622 non-null  int64 
 3   AnswerSequence1  89622 non-null  int64 
 4   AnswerSequence2  89622 non-null  int64 
 5   AnswerSequence3  89622 non-null  int64 
 6   TimeSpent        89622 non-null  int64 
 7   Test             89622 non-null  object
 8   TestKey          89622 non-null  int64 
dtypes: int64(8), object(1)
memory usage: 6.2+ MB


## Data Preparation

In [20]:
df = fca_question[["TestKey", "AnswerSequence1", "AnswerSequence2", "AnswerSequence3"]]
df.head(25)

,TestKey,AnswerSequence1,AnswerSequence2,AnswerSequence3
0,160065,5,4,2
1,160065,4,3,2
2,160065,4,2,3
3,160065,3,2,4
4,160065,2,5,1
5,160065,3,2,4
6,160065,4,3,2
7,160065,4,3,2
8,160065,2,3,4
9,160065,5,4,3


In [21]:
# Groepeer de data per CandidateId, en behoud ook InstanceId
all_test_answers = df.groupby(['TestKey']).agg({
    'AnswerSequence1': list,
    'AnswerSequence2': list,
    'AnswerSequence3': list
}).reset_index()

# Voeg alle antwoorden samen in een enkele array
all_test_answers['AllAnswers'] = all_test_answers.apply(
    lambda row: np.array(row['AnswerSequence1'] + row['AnswerSequence2'] + row['AnswerSequence3']),
    axis=1
)

# Behoud de kolommen CandidateId, InstanceId en AllAnswers
all_test_answers = all_test_answers[['TestKey', 'AllAnswers']]
all_test_answers

,TestKey,AllAnswers
0,160065,"[5, 4, 4, 3, 2, 3, 4, 4, 2, 5, 4, 3, 4, 4, 3, ..."
1,160066,"[2, 4, 5, 4, 4, 5, 2, 1, 5, 4, 4, 4, 3, 4, 3, ..."
2,160067,"[3, 1, 1, 2, 5, 4, 4, 3, 3, 3, 1, 4, 4, 5, 3, ..."
3,160068,"[2, 4, 3, 5, 5, 5, 4, 5, 4, 5, 2, 3, 5, 4, 5, ..."
4,160069,"[4, 4, 2, 1, 2, 4, 4, 4, 1, 2, 2, 4, 2, 2, 1, ..."
...,...,...
6889,166954,"[3, 5, 2, 5, 5, 2, 5, 5, 5, 5, 5, 5, 5, 4, 3, ..."
6890,166955,"[5, 3, 1, 3, 5, 5, 4, 2, 5, 2, 3, 3, 5, 4, 4, ..."
6891,166956,"[5, 2, 1, 5, 5, 5, 4, 3, 4, 3, 2, 2, 5, 1, 3, ..."
6892,166957,"[1, 2, 1, 3, 2, 1, 0, 0, 0, 0, 0, 0, 0, 2, 3, ..."


## Detection function

In [22]:
def detect_same_answers(TestKey, print_result=False):
    answers = all_test_answers[all_test_answers['TestKey'] == TestKey].reset_index().AllAnswers[0]

    ans_count = {
        0:0, 
        1:0, 
        2:0,
        3:0,
        4:0,
        5:0
        }
    total_count = 0

    for i in answers: 
        ans_count[i] += 1
        total_count += 1

    max_count = max(ans_count.values())
    most_common_ans = max(ans_count, key=ans_count.get)
    perc = (max_count/total_count) * 100
    if print_result:
        print(f"Candidate {TestKey} answered {most_common_ans} on {perc}% of the questions.")
    return perc


In [23]:
detect_same_answers(160068, print_result=True)

Candidate 160068 answered 4 on 30.76923076923077% of the questions.


30.76923076923077

In [24]:
detect_same_answers(160069, print_result=True)

Candidate 160069 answered 2 on 28.205128205128204% of the questions.


28.205128205128204

In [25]:
detect_same_answers(160070, print_result=True)

Candidate 160070 answered 5 on 25.64102564102564% of the questions.


25.64102564102564

## Exporting Data with Anomaly Check

In [26]:
all_test_answers['SameAnswerPercentage'] = all_test_answers['TestKey'].apply(detect_same_answers)
all_test_answers

,TestKey,AllAnswers,SameAnswerPercentage
0,160065,"[5, 4, 4, 3, 2, 3, 4, 4, 2, 5, 4, 3, 4, 4, 3, ...",30.769231
1,160066,"[2, 4, 5, 4, 4, 5, 2, 1, 5, 4, 4, 4, 3, 4, 3, ...",33.333333
2,160067,"[3, 1, 1, 2, 5, 4, 4, 3, 3, 3, 1, 4, 4, 5, 3, ...",28.205128
3,160068,"[2, 4, 3, 5, 5, 5, 4, 5, 4, 5, 2, 3, 5, 4, 5, ...",30.769231
4,160069,"[4, 4, 2, 1, 2, 4, 4, 4, 1, 2, 2, 4, 2, 2, 1, ...",28.205128
...,...,...,...
6889,166954,"[3, 5, 2, 5, 5, 2, 5, 5, 5, 5, 5, 5, 5, 4, 3, ...",30.769231
6890,166955,"[5, 3, 1, 3, 5, 5, 4, 2, 5, 2, 3, 3, 5, 4, 4, ...",25.641026
6891,166956,"[5, 2, 1, 5, 5, 5, 4, 3, 4, 3, 2, 2, 5, 1, 3, ...",30.769231
6892,166957,"[1, 2, 1, 3, 2, 1, 0, 0, 0, 0, 0, 0, 0, 2, 3, ...",53.846154


In [27]:
all_test_answers.drop(columns=["AllAnswers"], inplace=True)
all_test_answers

,TestKey,SameAnswerPercentage
0,160065,30.769231
1,160066,33.333333
2,160067,28.205128
3,160068,30.769231
4,160069,28.205128
...,...,...
6889,166954,30.769231
6890,166955,25.641026
6891,166956,30.769231
6892,166957,53.846154


In [28]:
all_test_answers['is_anomaly'] = all_test_answers.SameAnswerPercentage >= 50
all_test_answers.is_anomaly = all_test_answers.is_anomaly.astype(int)
all_test_answers.head()

,TestKey,SameAnswerPercentage,is_anomaly
0,160065,30.769231,0
1,160066,33.333333,0
2,160067,28.205128,0
3,160068,30.769231,0
4,160069,28.205128,0


In [29]:
all_test_answers.to_csv("../csv/same_answers_test_checked.csv")